Парсинг учебника

In [5]:
import os
import requests
import json
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm.notebook import tqdm
from PIL import Image
from io import BytesIO


BASE_DIR = "./"
DATA_DIR = os.path.join(BASE_DIR, "data")
IMG_DIR = os.path.join(DATA_DIR, "images")

os.makedirs(IMG_DIR, exist_ok=True)
print(f"Рабочая папка: {BASE_DIR}")
print(f"Картинки: {IMG_DIR}")

Рабочая папка: ./
Картинки: ./data/images


In [10]:
BASE_DOMAIN = "https://www.toehelp.ru"
URL_TEMPLATE = "https://www.toehelp.ru/theory/toe/lecture{0:02d}/lecture{0:02d}.html"
START_LEC = 1
END_LEC = 43

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}


def get_soup(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return None
        return BeautifulSoup(r.text, 'html.parser')
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None

def process_image(img_url, local_path_png):
    try:
        r = requests.get(img_url, headers=HEADERS, timeout=5)
        if r.status_code != 200: return False

        img_bytes = BytesIO(r.content)
        img = Image.open(img_bytes)
        img.seek(0) # Первый кадр

        # Фон для прозрачности
        bg = Image.new("RGB", img.size, (255, 255, 255))
        if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
            bg.paste(img.convert('RGBA'), mask=img.convert('RGBA'))
        else:
            bg.paste(img)

        bg.save(local_path_png, "PNG")
        return True
    except Exception as e:
        return False

def parse_lectures():
    theory_data = []
    images_data = []

    print(f"парсинг лекций {START_LEC}-{END_LEC}")

    for i in range(START_LEC, END_LEC + 1):
        lec_url = URL_TEMPLATE.format(i)
        lec_id = f"L{i:02d}"

        soup = get_soup(lec_url)
        if not soup: continue

        # хранение id последнего текстового блока
        last_text_id = None

        paragraphs = soup.find_all('p')

        for p_idx, p in enumerate(paragraphs):

            text = p.get_text(" ", strip=True)
            current_text_id = None

            # Если в параграфе есть текст
            if len(text) > 1:
                current_text_id = f"{lec_id}_p{p_idx}"
                theory_data.append({
                    "id": current_text_id,
                    "text": text,
                    "lecture_id": lec_id,
                    "source_url": lec_url
                })
                last_text_id = current_text_id

            # обработка картинки
            imgs = p.find_all('img')
            for img_idx, img in enumerate(imgs):
                src = img.get('src')
                if not src: continue

                # Логика URL
                if src.startswith("./theory"):
                    img_url = BASE_DOMAIN + src.lstrip('.')
                elif src.startswith("http"):
                    img_url = src
                else:
                    img_url = urljoin(lec_url, src)

                local_name = f"{lec_id}_p{p_idx}_img{img_idx}.png"
                local_path = os.path.join(IMG_DIR, local_name)

                # скачиваем (или пропускаем, если есть)
                if not os.path.exists(local_path):
                    success = process_image(img_url, local_path)
                    if not success: continue

                # Пытаемся найти подпись
                caption = img.get('alt', '')
                if not caption and len(text) < 300 and "Рис" in text:
                    # Если текст параграфа короткий и содержит "Рис", считаем его подписью
                    caption = text

                # Если в этом параграфе был текст -> привязываем к нему.
                # Если в этом параграфе только картинка -> привязываем к last_text_id
                linked_text_id = current_text_id if current_text_id else last_text_id

                images_data.append({
                    "id": f"{lec_id}_p{p_idx}_img{img_idx}",
                    "path": local_path,
                    "caption": caption,
                    "lecture_id": lec_id,
                    "preceding_text_id": linked_text_id
                })



    with open(os.path.join(DATA_DIR, "theory.json"), 'w', encoding='utf-8') as f:
        json.dump(theory_data, f, ensure_ascii=False, indent=4)

    with open(os.path.join(DATA_DIR, "images.json"), 'w', encoding='utf-8') as f:
        json.dump(images_data, f, ensure_ascii=False, indent=4)

    print(f"{len(theory_data)}")
    print(f"{len(images_data)} ")
    print(f"{DATA_DIR}")


In [11]:
parse_lectures()

парсинг лекций 1-43
Error fetching https://www.toehelp.ru/theory/toe/lecture04/lecture04.html: HTTPSConnectionPool(host='www.toehelp.ru', port=443): Read timed out. (read timeout=10)
Error fetching https://www.toehelp.ru/theory/toe/lecture14/lecture14.html: HTTPSConnectionPool(host='www.toehelp.ru', port=443): Read timed out. (read timeout=10)


Exception ignored in: <function tqdm.__del__ at 0x1069b02c0>
Traceback (most recent call last):
  File "/Users/sergey/Desktop/материалы для RAG/venv/lib/python3.13/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/Users/sergey/Desktop/материалы для RAG/venv/lib/python3.13/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


Error fetching https://www.toehelp.ru/theory/toe/lecture29/lecture29.html: HTTPSConnectionPool(host='www.toehelp.ru', port=443): Read timed out. (read timeout=10)
Error fetching https://www.toehelp.ru/theory/toe/lecture38/lecture38.html: HTTPSConnectionPool(host='www.toehelp.ru', port=443): Read timed out. (read timeout=10)
Error fetching https://www.toehelp.ru/theory/toe/lecture39/lecture39.html: HTTPSConnectionPool(host='www.toehelp.ru', port=443): Read timed out. (read timeout=10)
2128
2405 
./data


Структура Выходных Данных

Все данные сохраняются в папку: `Google Drive/EE_RAG/data/`.

### 📂 Файловая структура
```text
data/
├── images/               # Папка с изображениями
│   ├── L01_p3_img0.png   # Формат имени: {Лекция}_{Параграф}_{Индекс}.png
│   └── ...
├── theory.json           # Текстовый корпус
└── images.json           # Метаданные изображений и связи
```

### 📄 Формат JSON

#### 1. `theory.json` (Текстовая база знаний)
Содержит "чанки" текста (абзацы).
```json
[
  {
    "id": "L01_p4",                     // Уникальный ID параграфа
    "text": "В электрической цепи...",  // Очищенный текст
    "lecture_id": "L01",                // ID лекции
    "source_url": "https://..."         // Ссылка на источник
  },
  ...
]
```

#### 2. `images.json` (Мультимодальный индекс)
Содержит пути к файлам и **связь с текстом**.
```json
[
  {
    "id": "L01_p4_img0",
    "path": "/content/drive/.../L01_p4_img0.png", // Локальный путь
    "caption": "Рис. 1.2 Схема цепи",             // Подпись (если найдена)
    "preceding_text_id": "L01_p4",                // ID текста, объясняющего эту картинку
    "lecture_id": "L01"
  },
  ...
]
```
